# Lumpy 04: Planning and Agent Review

This notebook turns the final forecast reliability tiers into review queues, ownership and planning actions.


## 1. Load the operational output

We load the policy table written by the streamlined 02 notebook and keep the review rules visible.


In [ ]:
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, start.parent, start.parent.parent, start / "Inchscape Repo"]:
        if (candidate / "src" / "lumpy_forecasting.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate the Lumpy Fellas project root.")


project_root = find_project_root()
result_dir = project_root / "results/lumpy_02_forecasting"
review_dir = project_root / "docs/lumpy_02_forecasting"

policy = pd.read_csv(result_dir / "lumpy_02_operational_policy.csv")
policy_summary = pd.read_csv(review_dir / "lumpy_02_policy_summary.csv")

print("Project root:", project_root)
print("Policy rows:", len(policy))


## 2. Planning routes

We review the action, cadence, inventory treatment and owner assigned to each reliability tier.


In [ ]:
display(policy_summary[[
    "reliability_tier", "planning_action", "review_cadence", "inventory_treatment",
    "owner", "skus", "positive_skus", "underforecast_escalations"
]].sort_values("skus", ascending=False))


## 3. Underforecast escalation queue

We rank the manual and exception cases where forecast demand is below observed demand.


In [ ]:
escalations = policy.loc[policy.underforecast_escalation.astype(bool)].copy()
escalations["underforecast_units"] = (escalations.actual_18m - escalations.forecast_18m).clip(lower=0)
escalations = escalations.sort_values(["underforecast_units", "wmape"], ascending=[False, False])

print("Escalation SKUs:", len(escalations))
display(escalations[[
    "sku_id", "segment", "reliability_tier", "actual_18m", "forecast_18m",
    "underforecast_units", "wmape", "planning_action", "owner"
]].head(50).round(2))


## 4. Workload by segment

We group the review queue so ownership and meeting time can be planned around the largest problem areas.


In [ ]:
work_queue = (
    policy.groupby(["segment", "reliability_tier", "owner"], as_index=False)
    .agg(
        skus=("sku_id", "nunique"),
        actual_units=("actual_18m", "sum"),
        forecast_units=("forecast_18m", "sum"),
        underforecast_escalations=("underforecast_escalation", "sum"),
    )
    .sort_values(["underforecast_escalations", "skus"], ascending=False)
)
display(work_queue)


## 5. Review decision

We keep forecast-led SKUs in the normal monthly cycle and send the remaining tiers through the assigned review route.


In [ ]:
decision_table = pd.DataFrame([
    {"tier": "forecast_led", "decision": "Use as the planning forecast", "control": "Monitor monthly bias"},
    {"tier": "forecast_plus_review", "decision": "Use with monthly planner review", "control": "Check safety stock and lead time"},
    {"tier": "manual_review_with_forecast", "decision": "Use as guidance only", "control": "Approval required each planning cycle"},
    {"tier": "exception_policy", "decision": "Do not use point forecast alone", "control": "Event and inventory-policy review"},
    {"tier": "manual_lifecycle_review", "decision": "Route to lifecycle owner", "control": "Check supersession, obsolescence and reactivation"},
])
display(decision_table)
